# League of Legends Worlds 2026 — Swiss Stage & Beyond

This notebook creates the clean analysis dataset for the 2026 League of Legends World Championship using the Oracle's Elixir match dataset.

## Goal

1. Load the Oracle's Elixir 2026 match data.
2. Filter to the Worlds competition (`WLDs`).
3. Use October 23, 2026 as the start of the Swiss Stage.
4. Create the final Swiss Stage + subsequent Worlds dataset.
5. Validate that it contains the expected 72 unique games.
6. Save the cleaned dataset to `data/processed/`.

The original Oracle's Elixir rows are preserved; no player or team statistics are discarded at this stage.

## 1. Imports and project paths

In [1]:
from pathlib import Path
import pandas as pd

# Find the project root automatically.
# The project root contains the data/raw and data/processed folders.
current_dir = Path.cwd()
project_root = None

for folder in [current_dir, *current_dir.parents]:
    if (folder / "data" / "raw").exists() and (folder / "data" / "processed").exists():
        project_root = folder
        break

if project_root is None:
    raise FileNotFoundError(
        "Could not locate the project root. Make sure data/raw and data/processed exist."
    )

raw_dir = project_root / "data" / "raw"
processed_dir = project_root / "data" / "processed"

print("Project root:", project_root)
print("Raw data:", raw_dir)
print("Processed data:", processed_dir)

Project root: c:\Users\boyle\League_Of_Legends_Herald_Analysis
Raw data: c:\Users\boyle\League_Of_Legends_Herald_Analysis\data\raw
Processed data: c:\Users\boyle\League_Of_Legends_Herald_Analysis\data\processed


## 2. Load Oracle's Elixir data

In [2]:
# Oracle's Elixir source file
csv_path = raw_dir / "2026_LoL_esports_match_data_from_OraclesElixir.csv"

if not csv_path.exists():
    raise FileNotFoundError(f"Could not find Oracle's Elixir CSV: {csv_path}")

df = pd.read_csv(csv_path)

print("Dataset loaded successfully.")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Dataset loaded successfully.
Rows: 118848
Columns: 165


C:\Users\boyle\AppData\Local\Temp\ipykernel_16356\3170569646.py:7: DtypeWarning: Columns (0: url, 1: split) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


In [3]:
# Convert the date column once so it can be used reliably for filtering.
df["date"] = pd.to_datetime(df["date"], errors="coerce")

print("Date range:")
print(df["date"].min())
print(df["date"].max())

Date range:
2026-01-01 09:14:49
2026-11-10 09:49:05


## 3. Filter to Worlds 2026

In [4]:
# Oracle's Elixir uses the league code WLDs for the Worlds competition.
worlds_df = df[df["league"] == "WLDs"].copy()

print("Worlds rows:", len(worlds_df))
print("Unique Worlds games:", worlds_df["gameid"].nunique())
print("Worlds start:", worlds_df["date"].min())
print("Worlds end:", worlds_df["date"].max())

Worlds rows: 1272
Unique Worlds games: 106
Worlds start: 2026-09-17 09:11:42
Worlds end: 2026-11-09 00:56:00


### Worlds data validation

The Worlds dataset contains player-level and team-level records. The `position` column identifies the five player positions and the `team` record.

In [5]:
print("Positions:")
print(worlds_df["position"].value_counts(dropna=False).sort_index())

print("\nWorlds teams:")
print(sorted(worlds_df["teamname"].dropna().unique()))

Positions:
position
bot     212
jng     212
mid     212
sup     212
team    212
top     212
Name: count, dtype: int64

Worlds teams:
["Anyone's Legend", 'Bilibili Gaming', 'CTBC Flying Oyster', 'Cloud9', 'Dplus Kia', 'FURIA', 'G2 Esports', 'Gen.G', 'Hanwha Life Esports', 'Invictus Gaming', 'JD Gaming', 'Karmine Corp', 'LYON', 'LØS', 'MVK Esports', 'Movistar KOI', 'T1', 'Team Liquid', 'Team Secret Whales', 'Team WE', 'Top Esports']


## 4. Define the Swiss Stage start

The first 2026 Worlds Swiss Stage match was on **October 23, 2026**, between **Gen.G and Invictus Gaming**, on patch **16.20**.

Therefore, October 23 is used as the starting date for the analysis dataset. This excludes the earlier Worlds Play-In matches while retaining the Swiss Stage and subsequent Worlds matches.

In [6]:
SWISS_START = pd.Timestamp("2026-10-23")

worlds_swiss_onward = worlds_df[
    worlds_df["date"] >= SWISS_START
].copy()

print("Swiss + onward rows:", len(worlds_swiss_onward))
print("Swiss + onward unique games:", worlds_swiss_onward["gameid"].nunique())
print("First date:", worlds_swiss_onward["date"].min())
print("Last date:", worlds_swiss_onward["date"].max())

Swiss + onward rows: 864
Swiss + onward unique games: 72
First date: 2026-10-23 17:21:18
Last date: 2026-11-09 00:56:00


## 5. Validate the opening Swiss match

In [7]:
# The first Swiss match should be Gen.G vs. Invictus Gaming.
opening_match = worlds_swiss_onward[
    (worlds_swiss_onward["date"].dt.date == pd.Timestamp("2026-10-23").date())
                & (worlds_swiss_onward["position"] == "team")
            ].copy()

display(
    opening_match[
        ["gameid", "date", "game", "teamname", "side", "result", "patch", "gamelength"]
    ].sort_values(["date", "gameid", "side"])
)

,gameid,date,game,teamname,side,result,patch,gamelength
117106,LOLTMNT02_482269,2026-10-23 17:21:18,1,Invictus Gaming,Blue,0,16.2,1561
117107,LOLTMNT02_482269,2026-10-23 17:21:18,1,Gen.G,Red,1,16.2,1561
117118,LOLTMNT02_483314,2026-10-23 18:23:03,1,Top Esports,Blue,0,16.2,1431
117119,LOLTMNT02_483314,2026-10-23 18:23:03,1,Hanwha Life Esports,Red,1,16.2,1431
117130,LOLTMNT02_483342,2026-10-23 19:12:48,1,Movistar KOI,Blue,1,16.2,2321
117131,LOLTMNT02_483342,2026-10-23 19:12:48,1,Bilibili Gaming,Red,0,16.2,2321
117142,LOLTMNT02_482292,2026-10-23 20:16:06,1,Karmine Corp,Blue,1,16.2,1996
117143,LOLTMNT02_482292,2026-10-23 20:16:06,1,Anyone's Legend,Red,0,16.2,1996
117154,LOLTMNT02_482304,2026-10-23 21:16:28,1,Dplus Kia,Blue,1,16.2,1958
117155,LOLTMNT02_482304,2026-10-23 21:16:28,1,G2 Esports,Red,0,16.2,1958


## 6. Validate the final game count

In [8]:
expected_games = 72
actual_games = worlds_swiss_onward["gameid"].nunique()

print(f"Expected unique games: {expected_games}")
print(f"Actual unique games:   {actual_games}")

if actual_games != expected_games:
    raise ValueError(
        f"Expected {expected_games} games, but found {actual_games}. "
        "Review the date filter before continuing."
    )

print("Validation passed: 72 unique Worlds games found.")

Expected unique games: 72
Actual unique games:   72
Validation passed: 72 unique Worlds games found.


In [9]:
# Check the number of rows per game.
rows_per_game = worlds_swiss_onward.groupby("gameid").size()

print("Rows per game:")
print(rows_per_game.value_counts().sort_index())

print("\nUnique positions:")
print(sorted(worlds_swiss_onward["position"].dropna().unique()))

Rows per game:
12    72
Name: count, dtype: int64

Unique positions:
['bot', 'jng', 'mid', 'sup', 'team', 'top']


## 7. Save the clean dataset

This file preserves all Oracle's Elixir columns and all records belonging to the 72 Worlds games from the Swiss Stage onward.

In [10]:
output_path = processed_dir / "worlds_swiss_onward.csv"

worlds_swiss_onward.to_csv(output_path, index=False)

print(f"Saved dataset to: {output_path}")
print(f"File size: {output_path.stat().st_size / (1024**2):.2f} MB")

Saved dataset to: c:\Users\boyle\League_Of_Legends_Herald_Analysis\data\processed\worlds_swiss_onward.csv
File size: 0.68 MB


## Dataset complete

The cleaned Worlds dataset is now available at:

`data/processed/worlds_swiss_onward.csv`

### Current dataset definition

- Source: Oracle's Elixir 2026 match data
- Competition: Worlds (`WLDs`)
- Starting date: October 23, 2026
- Starting stage: Swiss Stage
- Unique games: 72
- Records: all original player and team records are retained

Future analysis should start from `worlds_swiss_onward` rather than modifying the original Oracle's Elixir data.